## Install Dependencies

In [1]:
# !pip install -q langchain==0.3.14
# !pip install -q langchain-openai==0.3.0
# !pip install -q langchain-community==0.3.14
# !pip install -q langgraph==0.2.64
# !pip install -q databricks-vectorsearch
# !pip install -U -qqqq databricks-langchain
# # !pip install -q jq

In [12]:
dbutils.library.restartPython()

## Step 1: Load the IT Support Knowledge Base

Load the IT support knowledge base from the JSON file. Each document contains:
- `text`: The IT support ticket description
- `steps`: Resolution steps (tool calls and actions)
- `metadata.ticket_type`: Either `workflow_jobs` or `log_lookup`

In [13]:
import json

with open("../docs/kb.json", "r") as f:
    knowledge_base = json.load(f)

print(f"Loaded {len(knowledge_base)} documents from knowledge base.")
knowledge_base[:3]

Loaded 15 documents from knowledge base.


[{'text': 'Please Refresh for Rural Master Sync in Cache Memory. REASON: Update in branch campaign Master & Field Allocation Master bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master',
  'steps': ["refresh_master_table('bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master')",
   'send_email_notification to stakeholders confirming refresh completion'],
  'metadata': {'ticket_type': 'workflow_jobs'}},
 {'text': 'Please update the below mentioned Table master - BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL. This update is required because some sources data was excluding because of source and internal source was not updated.',
  'steps': ["update_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL', 'Update source and internal source mappings')",
   "refresh_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL')",
   'send_email_notification to confirm update'],
  'metadata': {'ticket_type': 'workflow_jobs'}},
 {'text': 'Please sync the below tables bfl_std_lake.insurance_di

## Step 2: Create a Delta Table with Document Chunks

We convert the knowledge base documents into a Spark DataFrame and save it as a **Delta table** in Unity Catalog. This Delta table will serve as the source for the Vector Search index.

Each document's `text` and `steps` are combined into a single `content` field for richer embeddings.

| Component | Value |
|---|---|
| **Catalog** | `agentic_ai` |
| **Schema** | `langgraph` |
| **Table** | `it_support_kb_chunks` |
| **Columns** | `chunk_id` (INT), `content` (STRING), `ticket_type` (STRING) |

In [15]:
# Unity Catalog configuration
CATALOG = "agentic_ai"
SCHEMA = "langgraph"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.it_support_kb_chunks"

# Convert knowledge base to chunk data
# Combine ticket text with resolution steps for richer embeddings
chunk_data = []
for i, doc in enumerate(knowledge_base):
    # Build content from text + resolution steps
    steps_text = "\n".join(f"  {j+1}. {step}" for j, step in enumerate(doc.get("steps", [])))
    content = doc["text"]
    if steps_text:
        content += f"\n\nResolution Steps:\n{steps_text}"

    chunk_data.append({
        "chunk_id": i + 1,
        "content": content,
        "ticket_type": doc["metadata"]["ticket_type"]
    })

print(f"Created {len(chunk_data)} chunks.")
chunk_data[:3]

Created 15 chunks.


[{'chunk_id': 1,
  'content': "Please Refresh for Rural Master Sync in Cache Memory. REASON: Update in branch campaign Master & Field Allocation Master bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master\n\nResolution Steps:\n  1. refresh_master_table('bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master')\n  2. send_email_notification to stakeholders confirming refresh completion",
  'ticket_type': 'workflow_jobs'},
 {'chunk_id': 2,
  'content': "Please update the below mentioned Table master - BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL. This update is required because some sources data was excluding because of source and internal source was not updated.\n\nResolution Steps:\n  1. update_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL', 'Update source and internal source mappings')\n  2. refresh_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL')\n  3. send_email_notification to confirm update",
  'ticket_type': 'workflow_jobs'},
 {'chunk_id': 3,
  'content': "

In [5]:
# Create Spark DataFrame and save as Delta table
spark_df = spark.createDataFrame(chunk_data)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_NAME)

print(f"Delta table '{TABLE_NAME}' created successfully.")
display(spark.table(TABLE_NAME))

Delta table 'agentic_ai.langgraph.it_support_kb_chunks' created successfully.


,chunk_id,content,ticket_type
0,1,Please Refresh for Rural Master Sync in Cache Memory. REASON: Update in branch campaign Master & Field Allocation Master bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master\n\nResolution Steps:\n 1. refresh_master_table('bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master')\n 2. send_email_notification to stakeholders confirming refresh completion,workflow_jobs
1,2,"Please update the below mentioned Table master - BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL. This update is required because some sources data was excluding because of source and internal source was not updated.\n\nResolution Steps:\n 1. update_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL', 'Update source and internal source mappings')\n 2. refresh_master_table('BFL_STD_LAKE.PL_COE_MART.LEAD_ID_MASTER_SPL')\n 3. send_email_notification to confirm update",workflow_jobs
2,3,Please sync the below tables bfl_std_lake.insurance_distribution.pincode_health_master and bfl_std_lake.insurance_distribution.pb_campaign_master_rlr in RapidLR. Please refresh it after it refreshes at ICORS. Note: I have already raised the ticket for ICORS refresh (6743210)\n\nResolution Steps:\n 1. refresh_master_table('bfl_std_lake.insurance_distribution.pincode_health_master')\n 2. refresh_master_table('bfl_std_lake.insurance_distribution.pb_campaign_master_rlr')\n 3. send_email_notification confirming sync after ICORS refresh,workflow_jobs
3,4,"Please replace the Old default campaign by these new campaigns. Changes are only for PLPPL. Old Campaign: BFL_RAPIDLR_PL_EMERGING_HINDI_BARE, New Campaign: BFL_RLR_PLP_EME_AI_BOT_HIN_Pune. Old Campaign: BFL_RAPIDLR_PLCSG_P2_HINDI_BARE, New Campaign: BFL_RLR_PLP_GR_AI_BOT_HIN_Pune\n\nResolution Steps:\n 1. check_control_flags on campaign master to verify current mapping\n 2. update_master_table to replace old campaigns with new campaigns\n 3. refresh_master_table to sync campaign master\n 4. send_email_notification to confirm changes",workflow_jobs
4,5,"The PLPPL workflow pipeline is failing intermittently. The RapidLR PL pipeline for PLPPL offerProduct is throwing errors during the Business Rule Engine evaluation step. Need to investigate the EDS-RapidLR-PL repository code path and validate the master table configurations for plCtaControlMaster.\n\nResolution Steps:\n 1. run_pipeline_check('EDS-RapidLR-PL') to check pipeline health\n 2. check_control_flags('plCtaControlMaster', 'sendToSfdc, sfdcFlag')\n 3. update_master_table to fix flag misconfiguration if found\n 4. refresh_master_table to apply fixes\n 5. run_pipeline_check again to verify pipeline health\n 6. update_devops_ticket with RCA",workflow_jobs
5,6,"Since yesterday, no missed call feeds are flowing into Rapid LR. Source Name: bflslmscl2\n\nResolution Steps:\n 1. run_pipeline_check on ingestion pipeline\n 2. check_control_flags on source routing configuration\n 3. update_devops_ticket with findings\n 4. send_email_notification to alert source system team",workflow_jobs
6,7,As we checked the super app and web feeds are not flowing from 19th July. Kindly check.\n\nResolution Steps:\n 1. run_pipeline_check on ingestion pipeline for these sources\n 2. check_control_flags on source routing configuration\n 3. update_devops_ticket with detailed analysis\n 4. send_email_notification to escalate,workflow_jobs
7,8,"Please check the below sample logs ID, the data are not pushing on prospect staging of Growth SE business despite all masters have been updated. logId: 09508632-bbb4-45d8-8651-fbe8b044494e, a4150cd0-d148-4fef-96f0-1274b8635ca6, 1d776a1c-95c9-4ab3-9830-2b5f780a06d9\n\nResolution Steps:\n 1. query_log_view with the provided log IDs\n 2. Records show status=EXCLUDED, sfdcFlag=false\n 3. analyze_exclusion_reason for the excluded log IDs\n 4. check_control_flags('plCtaControlMaster', 'sendToSfdc, sfdcFlag')\n 5. update_master_table to set sfdcFlag=true for affected offerProduct/responseBusiness\n 6. refresh_master_table to sync changes\n 7. 

In [16]:
# Enable Change Data Feed (required for Delta Sync Index)
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("Change Data Feed enabled on the Delta table.")

Change Data Feed enabled on the Delta table.


## Step 3: Create a Vector Search Endpoint

A Vector Search **endpoint** is a compute resource that serves the vector search index. We create a `STANDARD` endpoint named `router_agent_endpoint`.

> **Note**: If the endpoint already exists, this cell will raise an error — you can safely skip it.

In [17]:
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME = "one-env-shared-endpoint-1"

vs_client = VectorSearchClient()

# Create the Vector Search endpoint (skip if it already exists)
try:
    vs_client.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print(f"Vector Search endpoint '{ENDPOINT_NAME}' created. It may take a few minutes to provision.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists. Skipping creation.")
    else:
        raise e

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Endpoint 'one-env-shared-endpoint-1' already exists. Skipping creation.


## Step 4: Create the Vector Search Index (with Managed Embeddings)

We create a **Delta Sync Index** on the Delta table. This index:
- Uses **`databricks-gte-large-en`** as the managed embedding model — Databricks automatically computes embeddings from the `content` column.
- Syncs automatically with the source Delta table via `TRIGGERED` pipeline.
- Uses `chunk_id` as the primary key.

> **Note**: If the index already exists, this cell will raise an error — you can safely skip it. The index creation may take several minutes to complete.

In [10]:
INDEX_NAME = f"{CATALOG}.{SCHEMA}.it_support_kb_index"

# Create the Delta Sync Index with managed embeddings
try:
    # Try to delete the index first if it exists on a different endpoint
    try:
        vs_client.delete_index(index_name=INDEX_NAME)
        print(f"Deleted existing index '{INDEX_NAME}' to recreate on correct endpoint.")
    except Exception:
        pass  # Index doesn't exist or already deleted
    
    index = vs_client.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        source_table_name=TABLE_NAME,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name="databricks-gte-large-en"
    )
    print(f"Vector Search index '{INDEX_NAME}' created successfully.")
    print("Note: Embedding computation may take a few minutes. Wait for the index to become ONLINE before querying.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{INDEX_NAME}' already exists. Skipping creation.")
    else:
        raise e

Vector Search index 'agentic_ai.langgraph.it_support_kb_index' created successfully.
Note: Embedding computation may take a few minutes. Wait for the index to become ONLINE before querying.


In [11]:
import time

# Wait for the index to become ONLINE
vs_index = vs_client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

status = vs_index.describe()
print(f"Index status: {status.get('status', {})}")

# Poll until the index is ready (optional - you can also check manually in the Databricks UI)
while status.get("status", {}).get("ready") != True:
    print("Index is not ready yet. Waiting 30 seconds...")
    time.sleep(30)
    status = vs_index.describe()
    print(f"Index status: {status.get('status', {})}")

print("Index is ONLINE and ready for queries!")

Index status: {'detailed_state': 'PROVISIONING_PIPELINE_RESOURCES', 'message': 'Index is currently pending setup of pipeline resources. Check latest status: https://e2-demo-field-eng.cloud.databricks.com/explore/data/agentic_ai/langgraph/it_support_kb_index', 'indexed_row_count': 0, 'provisioning_status': {'provisioning_pipeline_time_spent_seconds': 1.0}, 'ready': False, 'index_url': 'e2-demo-field-eng.cloud.databricks.com/api/2.0/vector-search/indexes/agentic_ai.langgraph.it_support_kb_index'}
Index is not ready yet. Waiting 30 seconds...
Index status: {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://e2-demo-field-eng.cloud.databricks.com/explore/data/agentic_ai/langgraph/it_support_kb_index', 'indexed_row_count': 13, 'triggered_update_status': {'last_processed_commit_version': 5, 'last_processed_commit_timestamp': '2026-02-19T22:35:09Z'}, 'ready': True, 'index_url': 'e2-demo-field-eng.cloud.databricks.com/api/2.0/vector

## Setup Environment Variables & LLM

We use **Databricks Foundation Model APIs** via `ChatDatabricks` so no external API keys are needed.

In [0]:
from databricks_langchain import ChatDatabricks

# Use a Databricks-hosted LLM endpoint
# Options: "databricks-claude-3-7-sonnet", "databricks-gpt-oss-120b", "databricks-meta-llama-3-3-70b-instruct"
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

## Step 5: Connect to the Vector Search Index

Now we connect to the Vector Search index we just created (or the pre-existing one) to use it for retrieval in our RAG pipeline.

In [15]:
# Get the vector search index (uses ENDPOINT_NAME and INDEX_NAME defined earlier)
vs_index = vs_client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
print(f"Connected to Vector Search Index: {INDEX_NAME}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Source Table: {TABLE_NAME}")

Connected to Vector Search Index: agentic_ai.langgraph.it_support_kb_index
Endpoint: one-env-shared-endpoint-1
Source Table: agentic_ai.langgraph.it_support_kb_chunks


## Helper: Convert Vector Search Results to LangChain Documents

Databricks Vector Search returns results in its own format. This helper converts them to LangChain `Document` objects for seamless integration with our RAG pipeline.

In [16]:
from langchain_core.documents import Document
from typing import List

def convert_vector_search_to_documents(results) -> List[Document]:
    """
    Convert Databricks Vector Search results to LangChain Document objects.
    
    The first column retrieved is loaded into page_content,
    and the rest (except the score column) into metadata.
    """
    column_names = [col["name"] for col in results["manifest"]["columns"]]
    
    langchain_docs = []
    for item in results["result"]["data_array"]:
        metadata = {}
        # Last element is the similarity score
        score = item[-1]
        # First element is page_content, middle elements are metadata
        for i in range(1, len(item) - 1):
            metadata[column_names[i]] = item[i]
        metadata["score"] = score
        doc = Document(page_content=item[0], metadata=metadata)
        langchain_docs.append(doc)
    
    return langchain_docs

## Test Vector Search Retrieval

Let's verify that the vector search index is working correctly with some sample queries and metadata filters.

In [17]:
# Test: Query with 'workflow_jobs' ticket_type filter
query = 'Please refresh master table for branch campaign'
results = vs_index.similarity_search(
    query_text=query,
    columns=["content", "ticket_type"],
    num_results=1,
    filters={"ticket_type": ["workflow_jobs"]},
    query_type="hybrid"
)
if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Score: 1.0000 | Ticket Type: workflow_jobs
Content: Please Refresh for Rural Master Sync in Cache Memory. REASON: Update in branch campaign Master & Field Allocation Master bfl_std_lake.rtl_cc.rpl_rapidlr_branch_campaign_master

Resolution Steps:
  1. ...
---


In [26]:
# Test: Query with 'workflow_jobs' ticket_type filter
query = 'Dummy Ticket'
results = vs_index.similarity_search(
    query_text=query,
    columns=["content", "ticket_type"],
    num_results=1,
    filters={"ticket_type": ["workflow_jobs"]},
    query_type="hybrid"
)
if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Score: 1.0000 | Ticket Type: workflow_jobs
Content: Please sync the below tables bfl_std_lake.insurance_distribution.pincode_health_master and bfl_std_lake.insurance_distribution.pb_campaign_master_rlr in RapidLR. Please refresh it after it refreshes a...
---


In [25]:
# Test: Query with 'log_lookup' ticket_type filter
results = vs_index.similarity_search(
    query_text="Data not pushing to prospect staging for Growth SE",
    columns=["content", "ticket_type"],
    num_results=1,
    filters={"ticket_type": ["log_lookup"]},
    query_type="ann"
)
print(results)
relevant_docs = convert_vector_search_to_documents(results)
retrieved_content = "\n\n".join(doc.page_content for doc in relevant_docs)

if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 3, 'columns': [{'name': 'content'}, {'name': 'ticket_type'}, {'name': 'score'}]}, 'result': {'row_count': 1, 'data_array': [["Please check the below sample logs ID, the data are not pushing on prospect staging of Growth SE business despite all masters have been updated. logId: 09508632-bbb4-45d8-8651-fbe8b044494e, a4150cd0-d148-4fef-96f0-1274b8635ca6, 1d776a1c-95c9-4ab3-9830-2b5f780a06d9\n\nResolution Steps:\n  1. query_log_view with the provided log IDs\n  2. Records show status=EXCLUDED, sfdcFlag=false\n  3. analyze_exclusion_reason for the excluded log IDs\n  4. check_control_flags('plCtaControlMaster', 'sendToSfdc, sfdcFlag')\n  5. update_master_table to set sfdcFlag=true for affected offerProduct/responseBusiness\n  6. refresh_master_table to sy

In [20]:
# Test: Query with 'workflow_jobs' ticket_type filter — pipeline issues
query = 'PLPPL workflow pipeline is failing intermittently'
results = vs_index.similarity_search(
    query_text=query,
    columns=["content", "ticket_type"],
    num_results=3,
    filters={"ticket_type": ["workflow_jobs"]},
    query_type="hybrid"
)

if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Score: 1.0000 | Ticket Type: workflow_jobs
Content: The PLPPL workflow pipeline is failing intermittently. The RapidLR PL pipeline for PLPPL offerProduct is throwing errors during the Business Rule Engine evaluation step. Need to investigate the EDS-Ra...
---
Score: 0.9761 | Ticket Type: workflow_jobs
Content: Since yesterday, no missed call feeds are flowing into Rapid LR. Source Name: bflslmscl2

Resolution Steps:
  1. check_upstream_source('bflslmscl2') to verify source status
  2. run_pipeline_check on ...
---
Score: 0.9612 | Ticket Type: workflow_jobs
Content: Please replace the Old default campaign by these new campaigns. Changes are only for PLPPL. Old Campaign: BFL_RAPIDLR_PL_EMERGING_HINDI_BARE, New Campaign: BFL_RLR_PLP_EME_AI_BOT_HIN_Pune. Old Campaig...
---


In [0]:
# Test: Query with 'log_lookup' ticket_type filter — exclusion issues
query = 'feeds are excluding as Default campaign not set'
results = vs_index.similarity_search(
    query_text=query,
    columns=["content", "ticket_type"],
    num_results=3,
    filters={"ticket_type": ["log_lookup"]},
    query_type="hybrid"
)

if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Score: 1.0000 | Category: log_lookup
Content: Question: What payment methods do you accept? Answer: We accept credit cards, PayPal, and wire transfers for corporate accounts....
---
Score: 0.9462 | Category: log_lookup
Content: Question: Can I get a quote for budgeting purposes? Answer: Yes, you can request a custom quote by contacting our sales team....
---
Score: 0.9340 | Category: log_lookup
Content: Question: Do you have a reseller program? Answer: Yes, we have a reseller program. Please contact our sales team for details....
---


In [0]:
# Test: Query without filter — find most similar ticket across all types
query = 'no missed call feeds are flowing into Rapid LR'
results = vs_index.similarity_search(
    query_text=query,
    columns=["content", "ticket_type"],
    num_results=3,
    query_type="hybrid"
)

if results["result"]["row_count"] == 0:
  print("No results found.")
else:
  docs = convert_vector_search_to_documents(results)
  for doc in docs:
      print(f"Score: {doc.metadata['score']:.4f} | Ticket Type: {doc.metadata['ticket_type']}")
      print(f"Content: {doc.page_content[:200]}...")
      print("---")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Score: 1.0000 | Category: workflow
Content: Question: What programming languages are supported by your SDK? Answer: Our SDK supports Python, Java, and JavaScript. Additional language support is ...
---
